# **Sentiment Analysis using NLP Pipeline & ML Models**

## **Objective**

The objective of this assignment is to build a complete end-to-end sentiment analysis system using NLP preprocessing, feature engineering, and multiple machine learning models. The performance of the models is evaluated and compared using standard metrics.

##Imports

In [2]:
import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


## Dataset Loading

In [27]:
df = pd.read_csv("IMDB Dataset.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## 1. Data Understanding

In [5]:
print("Columns in dataset:")
print(df.columns)

print("\nMissing values:")
print(df.isnull().sum())

print("\nClass distribution:")
print(df['sentiment'].value_counts())

print("\nSample text records:")
for i in range(3):
    print(f"\nSample {i+1}:")
    print(df['review'].iloc[i])

Columns in dataset:
Index(['review', 'sentiment'], dtype='object')

Missing values:
review       0
sentiment    0
dtype: int64

Class distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample text records:

Sample 1:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda.

## 2. NLP Preprocessing
Steps:
- Lowercasing
- Removing URLs
- Removing punctuation
- Removing numbers
- Removing special characters
- Tokenization
- Stopword removal
- Lemmatization

In [6]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text)

    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

In [7]:
df['cleaned_review'] = df['review'].apply(clean_text)

df[['review', 'cleaned_review']].head()

,review,cleaned_review
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


## Label Encoding

In [8]:
df['sentiment'] = df['sentiment'].str.lower()

label_map = {
    'negative': 0,
    'positive': 1
}

df['label'] = df['sentiment'].map(label_map)

df[['sentiment', 'label']].head()

,sentiment,label
0,positive,1
1,positive,1
2,positive,1
3,negative,0
4,positive,1


## 3. Feature Engineering
Using:
1. Bag of Words
2. TF-IDF

## Train-Test Split

In [18]:
X = df['cleaned_review']
y = df['label']

from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", len(X_train_text))
print("Test size :", len(X_test_text))

Train size: 40000
Test size : 10000


 Bag of Words


In [19]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train_text)
X_test_bow = bow.transform(X_test_text)

print("BoW Train Shape:", X_train_bow.shape)
print("BoW Test Shape :", X_test_bow.shape)

BoW Train Shape: (40000, 5000)
BoW Test Shape : (10000, 5000)


TF-IDF

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("TF-IDF Train Shape:", X_train_tfidf.shape)
print("TF-IDF Test Shape :", X_test_tfidf.shape)

TF-IDF Train Shape: (40000, 5000)
TF-IDF Test Shape : (10000, 5000)


## 4. Model Building
Models used:
- Logistic Regression
- Naive Bayes
- Decision Tree

In [21]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name, feature_type):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"\nModel: {model_name} ({feature_type})")
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(prec, 4))
    print("Recall   :", round(rec, 4))
    print("F1 Score :", round(f1, 4))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return [model_name, feature_type, acc, prec, rec, f1]

## 5 a). Model Evaluation using Bag of Words

In [22]:
results = []

lr = LogisticRegression()
nb = MultinomialNB()
dt = DecisionTreeClassifier(random_state=42)

results.append(evaluate_model(lr, X_train_bow, X_test_bow, y_train, y_test, "Logistic Regression", "BoW"))
results.append(evaluate_model(nb, X_train_bow, X_test_bow, y_train, y_test, "Naive Bayes", "BoW"))
results.append(evaluate_model(dt, X_train_bow, X_test_bow, y_train, y_test, "Decision Tree", "BoW"))


Model: Logistic Regression (BoW)
Accuracy : 0.8737
Precision: 0.8682
Recall   : 0.8812
F1 Score : 0.8746

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.87      0.87      5000
           1       0.87      0.88      0.87      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000


Model: Naive Bayes (BoW)
Accuracy : 0.8434
Precision: 0.8459
Recall   : 0.8398
F1 Score : 0.8428

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.85      0.84      5000
           1       0.85      0.84      0.84      5000

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84     10000


Model: Decision Tree (BoW)
Accuracy : 0.7204
Precision: 0.72
Recall   : 0.7212
F1 Score : 0.7

## 5 b). Model Evaluation using TF-IDF

In [23]:
results.append(evaluate_model(LogisticRegression(), X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf, "Logistic Regression", "TF-IDF"))
results.append(evaluate_model(MultinomialNB(), X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf, "Naive Bayes", "TF-IDF"))
results.append(evaluate_model(DecisionTreeClassifier(random_state=42), X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf, "Decision Tree", "TF-IDF"))


Model: Logistic Regression (TF-IDF)
Accuracy : 0.8862
Precision: 0.8788
Recall   : 0.896
F1 Score : 0.8873

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.88      0.89      5000
           1       0.88      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Model: Naive Bayes (TF-IDF)
Accuracy : 0.8527
Precision: 0.8471
Recall   : 0.8608
F1 Score : 0.8539

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.84      0.85      5000
           1       0.85      0.86      0.85      5000

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000


Model: Decision Tree (TF-IDF)
Accuracy : 0.7183
Precision: 0.7186
Recall   : 0.7176
F1 S

## 6 a). Model Comparison

In [24]:
results_df = pd.DataFrame(results, columns=[
    "Model", "Feature Type", "Accuracy", "Precision", "Recall", "F1 Score"
])

results_df = results_df.sort_values(by="F1 Score", ascending=False).reset_index(drop=True)

results_df

,Model,Feature Type,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,TF-IDF,0.8862,0.878776,0.8960,0.887304
1,Logistic Regression,BoW,0.8737,0.868177,0.8812,0.874640
2,Naive Bayes,TF-IDF,0.8527,0.847077,0.8608,0.853884
3,Naive Bayes,BoW,0.8434,0.845890,0.8398,0.842834
4,Decision Tree,BoW,0.7204,0.720048,0.7212,0.720624
5,Decision Tree,TF-IDF,0.7183,0.718606,0.7176,0.718103


## 6 b). Insights
- NLP preprocessing improved text quality significantly.
- TF-IDF generally gives better feature representation than BoW.
- Logistic Regression performs best for most text classification tasks.
- Naive Bayes is fast and effective.
- Decision Tree is interpretable but less accurate.

## Expected Pipeline Flow
Raw Data → Preprocessing → Feature Engineering → Model Training → Evaluation → Comparison

## Conclusion
In this assignment, a complete sentiment analysis pipeline was built. The text data was cleaned using NLP preprocessing techniques. Feature extraction was done using Bag of Words and TF-IDF. Multiple machine learning models were trained and evaluated. Based on the comparison, the best-performing model was identified. This assignment provided a clear understanding of how NLP and machine learning work together for sentiment analysis.

In [28]:
results_df = results_df.sort_values(by="F1 Score", ascending=False).reset_index(drop=True)

print("Best Model:")
print(results_df.iloc[0])

Best Model:
Model           Logistic Regression
Feature Type                 TF-IDF
Accuracy                     0.8862
Precision                  0.878776
Recall                        0.896
F1 Score                   0.887304
Name: 0, dtype: object
